In [ ]:
!pip -q install -U datasets evaluate seqeval transformers accelerate sentencepiece huggingface_hub

In [ ]:
import os, re, math, random, gc

import numpy as np

import torch
from torch import nn
import torch.nn.functional as F

from datasets import load_dataset, DatasetDict
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    pipeline
)

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix

from huggingface_hub import login
from google.colab import userdata

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:
ds = load_dataset("wikiann", "ru")
ds

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})

In [ ]:
train_size = 20000
val_size = 2000
test_size = 2000

ds_small = DatasetDict({
    "train": ds["train"].shuffle(seed=SEED).select(range(train_size)),
    "validation": ds["validation"].shuffle(seed=SEED).select(range(val_size)),
    "test": ds["test"].shuffle(seed=SEED).select(range(test_size)),
})
ds_small

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 2000
    })
})

In [ ]:
label_names = ds["train"].features["ner_tags"].feature.names
label_names

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']

### Подготовка данных

In [ ]:
def normalize_text(s):
    s = s.replace("\u00ad", "") # soft hyphen
    s = s.replace("\u200b", "") # zero-width space
    s = re.sub(r"\s+", " ", s).strip()
    return s

In [ ]:
def tokens_to_text_and_offsets(tokens):
    # Собираем текст как " ".join(tokens) и считаем char offsets каждого токена
    parts = []
    offsets = []
    cur = 0
    for i, t in enumerate(tokens):
        if i > 0:
            parts.append(" ")
            cur += 1
        start = cur
        parts.append(t)
        cur += len(t)
        end = cur
        offsets.append((start, end))
    text = "".join(parts)
    text = normalize_text(text)
    return text, offsets

In [ ]:
def bio_to_spans(labels):
    spans = []
    start = None
    ent_type = None

    for i, lab in enumerate(labels):
        if lab == "O":
            if ent_type is not None:
                spans.append((start, i, ent_type))
                start, ent_type = None, None
            continue

        prefix, typ = lab.split("-", 1)
        if prefix == "B":
            if ent_type is not None:
                spans.append((start, i, ent_type))
            start, ent_type = i, typ
        elif prefix == "I":
            if ent_type is None:
                # некорректная последовательность: считаем как B
                start, ent_type = i, typ
            elif typ != ent_type:
                spans.append((start, i, ent_type))
                start, ent_type = i, typ

    if ent_type is not None:
        spans.append((start, len(labels), ent_type))
    return spans

In [ ]:
def spans_char_from_bio(tokens, offsets, labels):
    # label-спаны в токенах - char-спаны
    spans = bio_to_spans(labels)
    char_spans = []
    for s, e, t in spans:
        cs = offsets[s][0]
        ce = offsets[e-1][1]
        char_spans.append((cs, ce, t))
    return char_spans

In [ ]:
def char_spans_to_bio(tokens, offsets, char_spans, label_set=("PER","ORG","LOC")):
    # Строгое отображение: токен должен полностью лежать внутри char-span.
    # Если span обрезает токен - это будет boundary error!
    labels = ["O"] * len(tokens)

    # сортировка по start, длиннее раньше (на случай вложенности)
    char_spans = sorted(char_spans, key=lambda x: (x[0], -(x[1]-x[0])))

    for (cs, ce, typ) in char_spans:
        if typ not in label_set:
            continue
        covered = []
        for i, (ts, te) in enumerate(offsets):
            if ts >= cs and te <= ce:
                covered.append(i)
        if not covered:
            continue
        labels[covered[0]] = "B-" + typ
        for i in covered[1:]:
            labels[i] = "I-" + typ

    return labels

### Утилиты для оценки качества

In [ ]:
seqeval_metric = evaluate.load("seqeval")

def seqeval_strict_micro_f1(y_true, y_pred):
    # evaluate/seqeval дает entity-level строгое совпадение спана
    # В res есть overall_f1/precision/recall
    res = seqeval_metric.compute(predictions=y_pred, references=y_true, zero_division=0)
    return res

In [ ]:
def boundary_error_breakdown(y_true, y_pred):
    # Анализ по спанам: exact, type_confusion, boundary_mismatch, missing, spurious
    stats = {
        "exact": 0,
        "type_confusion": 0,
        "boundary_mismatch": 0,
        "missing": 0,
        "spurious": 0,
        "total_gold": 0,
        "total_pred": 0,
    }

    for gt, pr in zip(y_true, y_pred):
        gsp = bio_to_spans(gt)
        psp = bio_to_spans(pr)

        stats["total_gold"] += len(gsp)
        stats["total_pred"] += len(psp)

        gset = {(s, e, t) for (s, e,  t) in gsp}
        pset = {(s, e, t) for (s, e, t) in psp}

        # Точные совпадения - одинаковые start, end и type
        exact = gset & pset
        stats["exact"] += len(exact)

        # Те же границы, разный тип
        gb = {(s, e): t for (s, e, t) in gsp}
        pb = {(s, e): t for (s, e, t) in psp}
        for b in set(gb.keys()) & set(pb.keys()):
            if gb[b] != pb[b]:
                stats["type_confusion"] += 1

        # boundary mismatch - есть пересечение по токенам, но точного совпадения по границам нет
        def overlaps(a, b):
            (s1, e1, _t1) = a
            (s2, e2, _t2) = b
            return max(s1, s2) < min(e1, e2)

        # Для каждой gold-сущности, которая не попала в exact:
        # если она пересекается хотя бы с одним предсказанным спаном, это boundary_mismatch
        # если не пересекается ни с одним, это missing
        for g in gsp:
            if g in exact:
                continue
            if any(overlaps(g, p) for p in psp):
                stats["boundary_mismatch"] += 1
            else:
                stats["missing"] += 1

        # Для каждой предсказанной сущности, которая не попала в exact:
        # если она не пересекается ни с одной gold-сущностью, это spurious
        # если пересекается, ничего не добавляется, потому что boundary_mismatch
        # уже считают только по gold-стороне, чтобы не удваивать счётчик
        for p in psp:
            if p in exact:
                continue
            if any(overlaps(p, g) for g in gsp):
                pass
            else:
                stats["spurious"] += 1

    return stats

In [ ]:
def confusion_matrix_by_type(y_true, y_pred, entity_types=("PER","ORG","LOC")):
    # Матрица только по exact boundary matches + type confusion (по одинаковым границам)
    # Классы: entity_types
    labels = list(entity_types)
    idx = {t:i for i,t in enumerate(labels)}
    cm = np.zeros((len(labels), len(labels)), dtype=int)

    for gt, pr in zip(y_true, y_pred):
        gsp = bio_to_spans(gt)
        psp = bio_to_spans(pr)
        gb = {(s,e): t for (s,e,t) in gsp}
        pb = {(s,e): t for (s,e,t) in psp}
        for b in set(gb.keys()) & set(pb.keys()):
            g = gb[b]
            p = pb[b]
            if g in idx and p in idx:
                cm[idx[g], idx[p]] += 1

    return cm, labels

def plot_confusion(cm, labels, title):
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
    plt.title(title)
    plt.xlabel("pred")
    plt.ylabel("gold")
    plt.show()

In [ ]:
def summarize(name, y_true, y_pred):
    res = seqeval_strict_micro_f1(y_true, y_pred)
    b = boundary_error_breakdown(y_true, y_pred)
    return {
        "model": name,
        "f1": res["overall_f1"],
        "precision": res["overall_precision"],
        "recall": res["overall_recall"],
        "exact_spans": b["exact"],
        "type_confusion": b["type_confusion"],
        "boundary_mismatch": b["boundary_mismatch"],
        "missing": b["missing"],
        "spurious": b["spurious"],
        "gold_spans": b["total_gold"],
        "pred_spans": b["total_pred"],
    }

rows = []

### Дистилляция

In [ ]:
base_model_id = "deepvk/RuModernBERT-base"
teacher_model_id = "masterkristall/rumodernbert_ner_ft"
student_id = "deepvk/RuModernBERT-small"

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

The `reference_compile` argument is deprecated and will be removed in `transformers v5.2.0`Use `torch.compile()` directly on the model instead.


In [ ]:
id2label = {i: l for i, l in enumerate(label_names)}
label2id = {l: i for i, l in enumerate(label_names)}

In [ ]:
teacher = AutoModelForTokenClassification.from_pretrained(
    teacher_model_id,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
).to(device)

for p in teacher.parameters():
    p.requires_grad = False

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

In [ ]:
def tokenize_and_align_labels(examples, max_length=256):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=max_length
    )

    aligned_labels = []
    for i in range(len(examples["tokens"])):
        word_ids = tokenized.word_ids(batch_index=i)
        gold = examples["ner_tags"][i]
        prev = None
        out = []
        for w in word_ids:
            if w is None:
                out.append(-100)
            elif w != prev:
                out.append(gold[w])
            else:
                # стандартно: продолжения слова игнорируем
                out.append(-100)
            prev = w
        aligned_labels.append(out)

    tokenized["labels"] = aligned_labels
    return tokenized

max_length = 192
tokenized = ds_small.map(lambda x: tokenize_and_align_labels(x, max_length=max_length), batched=True)
tokenized

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 2000
    })
})

In [ ]:
student = AutoModelForTokenClassification.from_pretrained(
    student_id,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
).to(device)

data_collator_s = DataCollatorForTokenClassification(tokenizer=tokenizer, padding="longest")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

ModernBertForTokenClassification LOAD REPORT from: deepvk/RuModernBERT-small
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
teacher.config.hidden_size, student.config.hidden_size

(768, 384)

In [ ]:
class DistillTrainer(Trainer):
    def __init__(
        self,
        teacher,
        temperature=2.0,
        alpha_kl=0.7,
        alpha_cos=0.3,          # вместо alpha_mse
        cos_warmup_frac=0.2,    # первые 20% шагов: 0 -> alpha_cos
        *args, **kwargs
    ):
        super().__init__(*args, **kwargs)
        self.teacher = teacher
        self.temperature = temperature
        self.alpha_kl = alpha_kl

        self.alpha_cos_target = alpha_cos
        self.cos_warmup_frac = cos_warmup_frac

        # проекция для hidden states
        tdim = self.teacher.config.hidden_size
        sdim = self.model.config.hidden_size
        self.model.kd_proj = nn.Linear(sdim, tdim, bias=False).to(self.model.device)

        self.teacher.eval()
        self.teacher.to(self.model.device)

    def _alpha_cos_current(self):
        # прогрев для cos части
        max_steps = getattr(self.state, "max_steps", None) or getattr(self.args, "max_steps", 0)
        if not max_steps or max_steps <= 0:
            return self.alpha_cos_target

        warmup_steps = max(1, int(self.cos_warmup_frac * max_steps))
        step = int(getattr(self.state, "global_step", 0))

        scale = min(1.0, step / warmup_steps)
        return self.alpha_cos_target * scale

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")

        # forward студента
        out_s = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs.get("attention_mask"),
            labels=labels,
            output_hidden_states=True,
            return_dict=True
        )
        loss_ce = out_s.loss

        #  forward учителя
        with torch.no_grad():
            out_t = self.teacher(
                input_ids=inputs["input_ids"],
                attention_mask=inputs.get("attention_mask"),
                output_hidden_states=True,
                return_dict=True
            )

        # Дистилляция логитов
        temp = self.temperature
        s_logits = out_s.logits / temp
        t_logits = out_t.logits / temp

        kl_mask = (labels != -100).view(-1)
        s_flat = s_logits.view(-1, s_logits.size(-1))[kl_mask]
        t_flat = t_logits.view(-1, t_logits.size(-1))[kl_mask]

        loss_kl = F.kl_div(
            F.log_softmax(s_flat, dim=-1),
            F.softmax(t_flat, dim=-1),
            reduction="batchmean"
        ) * (temp * temp)

        # Дистилляция hidden states
        hs_s = out_s.hidden_states
        hs_t = out_t.hidden_states

        student_depth = len(hs_s) - 1
        teacher_depth = len(hs_t) - 1

        loss_cos = 0.0

        if student_depth >= 2 and teacher_depth >= 2:
            # student слои 1..student_depth-1 без эмбеддингов=0 и без последнего слоя
            student_layers = list(range(1, student_depth))

            max_points = 3
            if len(student_layers) > max_points:
                positions = []
                for i in range(max_points):
                    pos = 1 + round(i * (student_depth - 2) / (max_points - 1))
                    positions.append(int(pos))
                student_layers = sorted(set(positions))

            # PKD-Skip сопоставление teacher-слоёв пропорционально по глубине
            teacher_layers = []
            for s_idx in student_layers:
                t_idx = int(round(s_idx * teacher_depth / student_depth))
                t_idx = max(1, min(teacher_depth - 1, t_idx))
                teacher_layers.append(t_idx)

            # маскирование как для KL: labels != -100 + attention_mask
            cos_mask = (labels != -100)
            if inputs.get("attention_mask") is not None:
                cos_mask = cos_mask & (inputs["attention_mask"].bool())
            cos_mask = cos_mask.to(dtype=out_s.logits.dtype)  # (B, L)

            denom = cos_mask.sum().clamp_min(1.0)

            for s_idx, t_idx in zip(student_layers, teacher_layers):
                a = self.model.kd_proj(hs_s[s_idx])
                b = hs_t[t_idx]

                a = F.normalize(a, p=2, dim=-1)
                b = F.normalize(b, p=2, dim=-1)

                cos_sim = (a * b).sum(dim=-1)
                token_loss = 1.0 - cos_sim

                loss_cos = loss_cos + (token_loss * cos_mask).sum() / denom

            loss_cos = loss_cos / len(student_layers)

        alpha_cos = self._alpha_cos_current()
        loss = loss_ce + self.alpha_kl * loss_kl + alpha_cos * loss_cos

        return (loss, out_s) if return_outputs else loss

In [ ]:
def compute_metrics_student(eval_pred):
    logits = eval_pred.predictions
    labels = eval_pred.label_ids

    if isinstance(logits, (tuple, list)):
        logits = logits[0]

    preds = np.argmax(logits, axis=-1)

    y_true, y_pred = [], []
    for p, l in zip(preds, labels):
        true_labels, pred_labels = [], []
        for pi, li in zip(p, l):
            if li == -100:
                continue
            true_labels.append(id2label[li])
            pred_labels.append(id2label[pi])
        y_true.append(true_labels)
        y_pred.append(pred_labels)

    res = seqeval_metric.compute(predictions=y_pred, references=y_true, zero_division=0)
    return {"f1": res["overall_f1"], "precision": res["overall_precision"], "recall": res["overall_recall"]}

In [ ]:
distill_args = TrainingArguments(
    output_dir="rumodernbert_small_distill",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    fp16=True,
    eval_strategy="steps",
    save_strategy="steps",
    logging_steps=50,
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    warmup_ratio=0.1,
    weight_decay=0.01
)


distill_trainer = DistillTrainer(
    teacher=teacher,
    model=student,
    args=distill_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator_s,
    compute_metrics=compute_metrics_student,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    temperature=3.0,
    alpha_kl=0.5,
    alpha_cos=0.1,
    cos_warmup_frac=0.2,
)

distill_trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Step,Training Loss,Validation Loss,F1,Precision,Recall
200,4.870065,2.023154,0.484096,0.428793,0.555778
400,2.624645,1.023940,0.679894,0.642500,0.721910
600,2.070771,0.933176,0.762268,0.727140,0.800963
800,1.529160,0.739289,0.780930,0.767028,0.795345
1000,1.181092,0.639614,0.812563,0.810132,0.815008
1200,1.262522,0.652262,0.808485,0.798981,0.818218
1400,0.824937,0.563621,0.832234,0.829414,0.835072
1600,0.817545,0.581352,0.836140,0.825832,0.846709
1800,0.769672,0.512545,0.844647,0.833268,0.856340
2000,0.525148,0.529742,0.837394,0.826750,0.848315


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3800, training_loss=1.439449140147159, metrics={'train_runtime': 1185.5124, 'train_samples_per_second': 168.703, 'train_steps_per_second': 5.272, 'total_flos': 448977995694816.0, 'train_loss': 1.439449140147159, 'epoch': 6.08})

In [ ]:
preds = distill_trainer.predict(tokenized["test"])
test_metrics = preds.metrics
test_metrics

{'test_loss': 0.4691792130470276,
 'test_f1': 0.843921250253704,
 'test_precision': 0.8316,
 'test_recall': 0.8566131025957973,
 'test_runtime': 12.6723,
 'test_samples_per_second': 157.825,
 'test_steps_per_second': 4.971}

In [ ]:
login(token=userdata.get("HF_WRITE"))

In [ ]:
distill_trainer.push_to_hub(commit_message="RuModernBERT-small-distilled ner finetuned")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...distill/model.safetensors:   0%|          |  556kB /  139MB            

  ...distill/training_args.bin:  75%|#######4  | 3.88kB / 5.20kB            

CommitInfo(commit_url='https://huggingface.co/masterkristall/rumodernbert_small_distill/commit/ddd5d99f6ee595f829d07ab5530e922300279f69', commit_message='RuModernBERT-small-distilled ner finetuned', commit_description='', oid='ddd5d99f6ee595f829d07ab5530e922300279f69', pr_url=None, repo_url=RepoUrl('https://huggingface.co/masterkristall/rumodernbert_small_distill', endpoint='https://huggingface.co', repo_type='model', repo_id='masterkristall/rumodernbert_small_distill'), pr_revision=None, pr_num=None)